In [ ]:
import os
from contextlib import closing
from pathlib import Path

import polars as pl
import psycopg2
from dotenv import load_dotenv

load_dotenv()

dsn = (
  f"host=localhost dbname={os.environ["EXAMPLE_DB"]} "
  f"user={os.environ["POSTGRES_USER"]} "
  f"password={os.environ["POSTGRES_PASSWORD"]}"
)

dsn_salesdb = (
    f"host=127.0.0.1 port=5432 dbname={os.environ["SALES_DB"]} "
    f"user={os.environ["POSTGRES_USER"]} "
    f"password={os.environ["POSTGRES_PASSWORD"]}"
)

In [ ]:
with closing(psycopg2.connect(dsn)) as conn:
    with conn:
        with conn.cursor() as cur:
            cur.execute(
"""
SELECT *
FROM customers;
"""
            )
            print(cur.fetchall())
            cur.execute(
"""
SELECT *
FROM orders;
"""
            )
            print(cur.fetchall())

In [ ]:
with closing(psycopg2.connect(dsn)) as conn:
    with conn:
        with conn.cursor() as cur:
            cur.execute(
"""
SELECT *
FROM customers c
INNER JOIN orders o
ON c.id = o.customer_id;
"""
            )
            print(cur.fetchall())

In [ ]:
sql = """
SELECT *
FROM customers c
INNER JOIN orders o
ON c.id = o.customer_id;
"""

with closing(psycopg2.connect(dsn)) as conn:
    pl.read_database(
        sql,
        conn,
    ).show()

In [ ]:
sql = """
SELECT *
FROM customers c
LEFT JOIN orders o
ON c.id = o.customer_id;
"""

with closing(psycopg2.connect(dsn)) as conn:
    pl.read_database(
        sql,
        conn,
    ).show()

In [ ]:
sql = """
SELECT *
FROM customers c
RIGHT JOIN orders o
ON c.id = o.customer_id;
"""

with closing(psycopg2.connect(dsn)) as conn:
    pl.read_database(
        sql,
        conn,
    ).show()

In [ ]:
sql = """
SELECT *
FROM customers c
FULL JOIN orders o
ON c.id = o.customer_id;
"""

with closing(psycopg2.connect(dsn)) as conn:
    df = pl.read_database(
        sql,
        conn,
    )
    print(df)

In [ ]:
sql = """
SELECT *
FROM customers c
LEFT JOIN orders o
ON c.id = o.customer_id
WHERE o.order_id IS NULL;
"""

with closing(psycopg2.connect(dsn)) as conn:
    df = pl.read_database(
        sql,
        conn,
    )
    print(df)

In [ ]:
sql = """
SELECT *
FROM customers c
RIGHT JOIN orders o
ON c.id = o.customer_id
WHERE c.id IS NULL;
"""

with closing(psycopg2.connect(dsn)) as conn:
    df = pl.read_database(
        sql,
        conn,
    )
    print(df)

In [ ]:
sql = """
SELECT *
FROM customers c
FULL JOIN orders o
ON c.id = o.customer_id
WHERE c.id       IS NULL
   OR o.order_id IS NULL;
"""

with closing(psycopg2.connect(dsn)) as conn:
    df = pl.read_database(
        sql,
        conn,
    )
    print(df)

In [ ]:
sql = """
SELECT *
FROM customers c
LEFT JOIN orders o
    ON c.id = o.customer_id
WHERE o.customer_id IS NOT NULL;
"""

with closing(psycopg2.connect(dsn)) as conn:
    df = pl.read_database(
        sql,
        conn,
    )
    print(df)

In [ ]:
with closing(psycopg2.connect(dsn_salesdb)) as conn:
    with conn:
        with conn.cursor() as cur:
            cur.execute("SELECT 1")
            print(cur.fetchall())

In [ ]:
sql = """
SELECT
    o.orderid
  , CONCAT_WS(' ', c.firstname, c.lastname) AS customer_name
  , p.product                               AS product_name
  , o.quantity * o.sales                    AS sales_amount
  , p.price                                 AS product_price
  , CONCAT_WS(' ', e.firstname, e.lastname) AS salesperson_name
FROM sales.orders o
LEFT JOIN sales.customers c
    ON o.customerid = c.customerid
LEFT JOIN sales.products p
    ON o.productid = p.productid
LEFT JOIN sales.employees e
    ON o.salespersonid = e.employeeid
ORDER BY o.orderid;
"""

with closing(psycopg2.connect(dsn_salesdb)) as conn:
    df = pl.read_database(sql, conn)
    print(df)

In [ ]:
# LeetCode Section

def run_problem(prob_num: str) -> pl.DataFrame:
    problem_dpath = Path.cwd() / "sql" / "leetcode" / prob_num
    setup_script = problem_dpath / "setup.sql"
    solution_script = problem_dpath / "solution.sql"
    dsn = f"""
host=127.0.0.1 dbname={os.environ["LEETCODE_DB"]}
user={os.environ["POSTGRES_USER"]} password={os.environ["POSTGRES_PASSWORD"]}
"""
    with closing(psycopg2.connect(dsn)) as conn:
        with conn, conn.cursor() as cur:
            cur.execute(setup_script.read_text())
            print(f"data for {prob_num} prepared:", cur.rowcount)
        
        with conn:
            return pl.read_database(solution_script.read_text(), conn)

In [ ]:
# p1193
run_problem("p1193")

In [ ]:
# p1174
run_problem("p1174")

In [ ]:
# p550
run_problem("p550")

In [ ]:
# p1070
run_problem("p1070")

In [ ]:
# p1045
run_problem("p1045")

In [ ]:
# p180
run_problem("p180")

In [ ]:
# p1164
run_problem("p1164")

In [ ]:
from datetime import date

dsn = f"""
host=127.0.0.1 dbname={os.environ["LEETCODE_DB"]}
user={os.environ["POSTGRES_USER"]} password={os.environ["POSTGRES_PASSWORD"]}
"""

with closing(psycopg2.connect(dsn)) as conn:
    products_df = pl.read_database("SELECT * FROM p1164.Products;", conn)

result = (
    products_df
    .select(
        pl.col("product_id").unique(),
        pl.lit(date(2019, 8, 16)).alias("target_dt"),
    )
    .join_asof(
        products_df.sort(by="change_date"),
        right_on="change_date",
        left_on="target_dt",
        by="product_id",
    )
    .select(
        "product_id",
        pl.col("new_price").fill_null(10).alias("price"),
    )
)

print(result)

In [ ]:
# p1204
run_problem("p1204")

In [ ]:
# p1907
run_problem("p1907")

## Rules of Set Operations

1. `ORDER BY` can be used only once
2. Same number of columns
3. Matching Data Types
4. Same Order of Columns
5. First Query Controls Aliases
6. Mapping Correct Columns

In [ ]:
sql = """
SELECT
    firstname
  , lastname
FROM sales.employees

UNION

SELECT
    firstname
  , lastname
FROM sales.customers
;
"""

with closing(psycopg2.connect(dsn_salesdb)) as conn:
    df = pl.read_database(sql, conn)

print(df)

In [ ]:
sql = """
SELECT
    firstname
  , lastname
FROM sales.employees

UNION ALL

SELECT
    firstname
  , lastname
FROM sales.customers
;
"""

with closing(psycopg2.connect(dsn_salesdb)) as conn:
    df = pl.read_database(sql, conn)

print(df)

In [ ]:
sql = """
SELECT
    firstname
  , lastname
FROM sales.employees
EXCEPT
SELECT
    firstname
  , lastname
FROM sales.customers
;
"""

with closing(psycopg2.connect(dsn_salesdb)) as conn:
    df = pl.read_database(sql, conn)

print(df)

In [ ]:
sql = """
SELECT
    firstname
  , lastname
FROM sales.employees
INTERSECT
SELECT
    firstname
  , lastname
FROM sales.customers
;
"""

with closing(psycopg2.connect(dsn_salesdb)) as conn:
    df = pl.read_database(sql, conn)

print(df)

In [ ]:
sql = """
SELECT
    'orders' AS data_src
  , *
FROM sales.orders
UNION
SELECT
    'ordersarchive'
  , *
FROM sales.ordersarchive
ORDER BY orderid
;
"""

with closing(psycopg2.connect(dsn_salesdb)) as conn:
    df = pl.read_database(sql, conn)

df